# 🧪 LAB: Manual MLPs for Classification and Regression

In this lab, you will use `PyTorch` to implement manually a multi-layer perceptron (MLP) for three different tasks: binary classification, multi-class classification and regression. 

## General instructions to complete in ALL three tasks:

1. ***IMPLEMENTATION***:
   
Implement a separate class for each task:

  - `BinaryMLP` for the binary classification task  
  - `MultiClassMLP` for the multi-class classification task  
  - `RegressionMLP` for the regression task

   Each class must include the following methods:

  - `__init__` for initializing 1 or 2 hidden layers. (*Help*: having a second hidden layer only means the need of having a extra set of weights and intercept). 
  - `forward` to transfer information from the input to the output layer. (*Help*: make sure that information passes through a second hidden layer if there is one).
  - `cost` computing the cost.
  - `fit` for training, using the gradients to manually update the parameters of the model. **Use stochastic gradient descent to update your weights**. *N.B.* You may probably reuse much of the code we used in this week tutorial already. 
  - `predict` to convert the information at the output layer into the required output.

2. ***DATA PREPARATION***

For each task, you will be provided with a toy dataset. For each dataset:

  - Split into training and holdout sets (use an 80/20 split)
  - Standardize the features properly, avoiding data leakage
  - Convert all data into `PyTorch` Tensors for compatibility

3. ***MODEL TRAINING AND EVALUATION***

Instantiate the model for each task and train it under different hyperparameter configurations. In each case, record the performance on **both** the training and holdout sets using the appropriate metric for the task (accuracy, MSE, etc.). You should explore the following configurations:

  - One hidden layer, varying the number of hidden units (use ReLU as their activation function)
  - Same number of hidden units across one or two hidden layers (use ReLU as their activation function)
  - Repeat the above setups using Tanh activation instead of ReLU

Present your results in a compact way (e.g., a summary table, a data frame etc).

**NOTE**: When training your model, use a fixed learning rate of your choice (e.g., 0.01 is a reasonable starting point) and a reasonably large number of  epochs (e.g., 100–200) based on how training and holdout performance evolve.

4. ***REFLECTION AND DISCUSSION***

Reflect on the impact of the different hyperparameter settings:

- How does the number of hidden units affect performance?
- What changes when using two layers instead of one?
- How does the activation function (ReLU vs. Tanh) influence results?

Please elaborate your answers.

---

**Collaboration Note**: This assignment is designed to support collaborative work. We encourage you to divide tasks among group members so that everyone can contribute meaningfully. Many components of the assignment can be approached in parallel or split logically across team members. Good coordination and thoughtful integration of your work will lead to a stronger final result.

**Ideally, each group member should be responsible for one of the separate tasks.** BUT, everyone should help each other along the way, both reviewing and refining results and discussion.

---

In total, this lab assignment will be worth **100 points**.

--- 
**Submission notes**:

* Write down all group members' names, or at least the group name (if you have one and you previously provided it), in the first cell of the notebook.

* Verify that the notebook runs as expected and that all required outputs are included.


NAME(s) = ""

## 1. Pre-implementation Group Discussion (15 points)

Discuss and agree on:

- What cost function should be used for each of the below task.
- What changes are needed in the output layer for each of these tasks. In particular, consider the number of units and the activation function.
- Why it is important to standardize the data before training each model.
- How you could detect overfitting when training your models.

USE AS MANY MARKDOWN CELLS AS NEEDED

## 2. Binary Classification (25 points)

Use the dataset below to complete points 1 to 4 in the general instructions for this task.

Use as many cells as needed.

In [ ]:
from sklearn.datasets import make_moons

X, y = make_moons(n_samples=1000, noise=0.2, random_state=42)

In [ ]:
# USE AS MANY CELLS AS NEEDED

## 2. Multi-Class Classification (25 points)

Use the dataset below to complete points 1 to 4 in the general instructions for this task.

Use as many cells as needed.

---
**NOTE**: This will likely be the most challenging exercise. To help you, here are some pointers:

 - The **output (last) layer** should have as many units as there are classes in your data.  

 - The **activation function of the output layer must be softmax**, not sigmoid. Softmax ensures that all output values are between 0 and 1 and sum to 1, so they can be interpreted as probabilities across the different classes. (*Help*: you may want to look into `torch.nn.functional` to find its implementation...)

 - For a multi-class problem, your target variable `y` should be **one-hot encoded**. For example:  
   - Label = 0 → [1, 0, 0]  
   - Label = 1 → [0, 1, 0]  
   - Label = 2 → [0, 0, 1]  
   You can easily achieve this with `OneHotEncoder` from `sklearn.preprocessing`.  

 - The **predicted class** corresponds to the unit with the highest probability.  
   Example:  
   - `[0.1, 0.3, 0.6] → class 2`  
   - `[0.6, 0.2, 0.2] → class 0`  

 - For this exercise, you need to **implement the categorical cross-entropy loss** (an extension of binary cross-entropy to multiple classes). It is defined as:  

   $$\sum_{c=1}^{l} y_{o,c}\,\log(p_{o,c}),$$

   where $l$ is the number of classes, $y_{o,c}$ is the one-hot encoded label for observation $o$, and $p_{o,c}$ is the predicted probability for class $c$ (after applying softmax). The log is the natural logarithm.

---

In [5]:
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=1000, n_classes=3, 
                           n_clusters_per_class=1, 
                           n_features=2, n_informative=2, n_redundant=0, random_state=1234, flip_y=0.15)

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn.functional as F

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1234)
standardizer = StandardScaler()
X_train_std = standardizer.fit_transform(X_train)
X_test_std = standardizer.transform(X_test)

X_train_std = torch.tensor(X_train_std, dtype=torch.float32)
X_test_std = torch.tensor(X_test_std, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

In [7]:
class MulticlassClassification_1layer:
    def __init__(self, hidden_units, activation_func, num_features = 2, num_classes = 3, learning_rate = 0.01):
        self.num_features = num_features
        self.num_classes = num_classes
        self.hidden_units = hidden_units
        self.activation_func = activation_func
        self.lr = learning_rate
        self.activation_function = torch.relu if activation_func == "relu" else torch.tanh

        self.W_1 = torch.randn(num_features, hidden_units, requires_grad = True)
        self.b_1 = torch.randn(hidden_units, requires_grad = True)
        self.W_2 = torch.randn(hidden_units, num_classes, requires_grad = True)
        self.b_2 = torch.randn(num_classes, requires_grad = True)
    
    def forward(self, X):
        h = torch.matmul(X, self.W_1) + self.b_1
        h_act = self.activation_function(h)
        out = torch.matmul(h_act, self.W_2) + self.b_2
        return torch.softmax(out, dim=1)

    def cost(self, y_hat, y_train_oh):
        eps = 1e-12
        return -(y_train_oh * torch.log(y_hat + eps)).sum(dim=1).mean()
    
    def fit(self, X, y, n_epochs = 100):
        X = torch.as_tensor(X, dtype=torch.float32)
        y = torch.as_tensor(y, dtype=torch.long)
        y_train_oh = torch.nn.functional.one_hot(y, num_classes=self.num_classes).float()

        for epoch in range(n_epochs):
            randperm = torch.randperm(X.shape[0])
            for ii in randperm:
                x_batch = X[ii:ii+1]             
                y_batch = y_train_oh[ii:ii+1]
                probs = self.forward(x_batch)
                cost = self.cost(probs, y_batch)
                cost.backward()

                with torch.no_grad():
                    self.W_1 -= self.lr * self.W_1.grad
                    self.b_1 -= self.lr * self.b_1.grad
                    self.W_2 -= self.lr * self.W_2.grad
                    self.b_2 -= self.lr * self.b_2.grad

                    # Zero gradients
                    self.W_1.grad.zero_()
                    self.b_1.grad.zero_()
                    self.W_2.grad.zero_()
                    self.b_2.grad.zero_()
            if epoch % 10 == 0:
                print(f"Epoch {epoch}, Cost: {cost.item():.4f}")
        return self

    def predict(self, X):
        X = torch.as_tensor(X, dtype=torch.float32)
        with torch.no_grad():
            return self.forward(X).argmax(dim=1).numpy()

In [11]:
class MulticlassClassification_2layer:
    def __init__(self, hidden_units, activation_func, num_features = 2, num_classes = 3, learning_rate = 0.01):
        self.num_features = num_features
        self.num_classes = num_classes
        self.hidden_units = hidden_units
        self.activation_func = activation_func
        self.lr = learning_rate
        self.activation_function = torch.relu if activation_func == "relu" else torch.tanh

        self.W_1 = torch.randn(num_features, hidden_units, requires_grad = True)
        self.b_1 = torch.randn(hidden_units, requires_grad = True)
        self.W_2 = torch.randn(hidden_units, hidden_units, requires_grad = True)
        self.b_2 = torch.randn(hidden_units, requires_grad = True)
        self.W_3 = torch.randn(hidden_units, num_classes, requires_grad = True)
        self.b_3 = torch.randn(num_classes, requires_grad = True)

    
    def forward(self, X):
        h1 = torch.matmul(X, self.W_1) + self.b_1
        h1_act = self.activation_function(h1)
        h2 = torch.matmul(h1_act, self.W_2) + self.b_2
        h2_act = self.activation_function(h2)
        out = torch.matmul(h2_act, self.W_3) + self.b_3
        return torch.softmax(out, dim=1)

    def cost(self, y_hat, y_train_oh):
        eps = 1e-12
        return -(y_train_oh * torch.log(y_hat + eps)).sum(dim=1).mean()
    
    def fit(self, X, y, n_epochs = 100):
        X = torch.as_tensor(X, dtype=torch.float32)
        y = torch.as_tensor(y, dtype=torch.long)
        y_train_oh = torch.nn.functional.one_hot(y, num_classes=self.num_classes).float()

        for epoch in range(n_epochs):
            randperm = torch.randperm(X.shape[0])
            for ii in randperm:
                x_batch = X[ii:ii+1]             
                y_batch = y_train_oh[ii:ii+1]
                probs = self.forward(x_batch)
                cost = self.cost(probs, y_batch)
                cost.backward()

                with torch.no_grad():
                    self.W_1 -= self.lr * self.W_1.grad
                    self.b_1 -= self.lr * self.b_1.grad
                    self.W_2 -= self.lr * self.W_2.grad
                    self.b_2 -= self.lr * self.b_2.grad
                    self.W_3 -= self.lr * self.W_3.grad
                    self.b_3 -= self.lr * self.b_3.grad

                    # Zero gradients
                    self.W_1.grad.zero_()
                    self.b_1.grad.zero_()
                    self.W_2.grad.zero_()
                    self.b_2.grad.zero_()
                    self.W_3.grad.zero_()
                    self.b_3.grad.zero_()
            if epoch % 10 == 0:
                print(f"Epoch {epoch}, Cost: {cost.item():.4f}")
        return self

    def predict(self, X):
        X = torch.as_tensor(X, dtype=torch.float32)
        with torch.no_grad():
            return self.forward(X).argmax(dim=1).numpy()

In [ ]:
# 1 layer relu with differing number of hidden units
import pandas as pd 

hidden_units_grid = [2, 4, 8, 16, 32, 64]
results = []
epoches = 100
for units in hidden_units_grid:
    torch.manual_seed(1234)
    model = MulticlassClassification_1layer(units, "relu")
    model.fit(X_train_std, y_train, n_epochs= epoches)

    train_acc = (model.predict(X_train_std) == y_train.numpy()).mean()
    holdout_acc = (model.predict(X_test_std) == y_test.numpy()).mean()

    results.append({"layers": 1, "units": units, "activation": "relu",
                    "train_acc": round(float(train_acc), 3),
                    "holdout_acc": round(float(holdout_acc), 3)})

df_relu_1layer = pd.DataFrame(results)
df_relu_1layer

Epoch 0, Cost: 0.8925
Epoch 10, Cost: 0.6882
Epoch 20, Cost: 0.0945
Epoch 30, Cost: 0.1237
Epoch 40, Cost: 0.1798
Epoch 50, Cost: 0.0520
Epoch 60, Cost: 0.0249
Epoch 70, Cost: 0.8306
Epoch 80, Cost: 0.7611
Epoch 90, Cost: 2.0313
Epoch 0, Cost: 0.2553
Epoch 10, Cost: 0.1571
Epoch 20, Cost: 0.1878
Epoch 30, Cost: 0.9672
Epoch 40, Cost: 1.3768
Epoch 50, Cost: 0.1943
Epoch 60, Cost: 0.1936
Epoch 70, Cost: 5.4150
Epoch 80, Cost: 0.2331
Epoch 90, Cost: 0.1323
Epoch 0, Cost: 0.6218
Epoch 10, Cost: 0.0795
Epoch 20, Cost: 0.0774
Epoch 30, Cost: 0.0383
Epoch 40, Cost: 1.4198
Epoch 50, Cost: 0.1351
Epoch 60, Cost: 0.2671
Epoch 70, Cost: 0.1067
Epoch 80, Cost: 0.9672
Epoch 90, Cost: 0.0887
Epoch 0, Cost: 0.1237
Epoch 10, Cost: 0.1032
Epoch 20, Cost: 1.0640
Epoch 30, Cost: 0.3556
Epoch 40, Cost: 0.5973
Epoch 50, Cost: 0.0903
Epoch 60, Cost: 1.1143
Epoch 70, Cost: 1.5393
Epoch 80, Cost: 0.0667
Epoch 90, Cost: 0.4857
Epoch 0, Cost: 3.3033
Epoch 10, Cost: 0.0817
Epoch 20, Cost: 0.1161
Epoch 30, Cost: 

,layers,units,activation,train_acc,holdout_acc
0,1,2,relu,0.647,0.610
1,1,4,relu,0.777,0.795
2,1,8,relu,0.807,0.820
3,1,16,relu,0.802,0.800
4,1,32,relu,0.809,0.815
5,1,64,relu,0.818,0.825


In [10]:
# 1 layer tanh differing number of hidden units

hidden_units_grid = [2, 4, 8, 16, 32, 64]
results = []
epoches = 100
for units in hidden_units_grid:
    torch.manual_seed(1234)
    model = MulticlassClassification_1layer(units, "tanh")
    model.fit(X_train_std, y_train, n_epochs= epoches)

    train_acc = (model.predict(X_train_std) == y_train.numpy()).mean()
    holdout_acc = (model.predict(X_test_std) == y_test.numpy()).mean()

    results.append({"layers": 1, "units": units, "activation": "tanh",
                    "train_acc": round(float(train_acc), 3),
                    "holdout_acc": round(float(holdout_acc), 3)})

df_tanh_1layer = pd.DataFrame(results)
df_tanh_1layer

Epoch 0, Cost: 0.1432
Epoch 10, Cost: 0.2969
Epoch 20, Cost: 0.1228
Epoch 30, Cost: 0.1362
Epoch 40, Cost: 0.1268
Epoch 50, Cost: 0.1034
Epoch 60, Cost: 0.1056
Epoch 70, Cost: 0.1135
Epoch 80, Cost: 0.1351
Epoch 90, Cost: 2.4796
Epoch 0, Cost: 0.2405
Epoch 10, Cost: 0.2179
Epoch 20, Cost: 0.0975
Epoch 30, Cost: 0.4360
Epoch 40, Cost: 0.8084
Epoch 50, Cost: 0.1116
Epoch 60, Cost: 0.1075
Epoch 70, Cost: 2.6852
Epoch 80, Cost: 0.1498
Epoch 90, Cost: 0.1490
Epoch 0, Cost: 0.7191
Epoch 10, Cost: 0.1225
Epoch 20, Cost: 0.0950
Epoch 30, Cost: 0.1293
Epoch 40, Cost: 1.6541
Epoch 50, Cost: 0.1467
Epoch 60, Cost: 0.2947
Epoch 70, Cost: 0.1065
Epoch 80, Cost: 0.9900
Epoch 90, Cost: 0.0673
Epoch 0, Cost: 0.1628
Epoch 10, Cost: 0.1033
Epoch 20, Cost: 1.2644
Epoch 30, Cost: 0.2767
Epoch 40, Cost: 0.6034
Epoch 50, Cost: 0.0772
Epoch 60, Cost: 1.1439
Epoch 70, Cost: 1.6229
Epoch 80, Cost: 0.1014
Epoch 90, Cost: 0.3056
Epoch 0, Cost: 3.8433
Epoch 10, Cost: 0.1145
Epoch 20, Cost: 0.1242
Epoch 30, Cost: 

,layers,units,activation,train_acc,holdout_acc
0,1,2,tanh,0.794,0.820
1,1,4,tanh,0.794,0.805
2,1,8,tanh,0.809,0.830
3,1,16,tanh,0.809,0.830
4,1,32,tanh,0.807,0.825
5,1,64,tanh,0.819,0.830


In [12]:
# 2 layer relu differing number of hidden units

hidden_units_grid = [2, 4, 8, 16, 32, 64]
results = []
epoches = 100
for units in hidden_units_grid:
    torch.manual_seed(1234)
    model = MulticlassClassification_2layer(units, "relu")
    model.fit(X_train_std, y_train, n_epochs= epoches)

    train_acc = (model.predict(X_train_std) == y_train.numpy()).mean()
    holdout_acc = (model.predict(X_test_std) == y_test.numpy()).mean()

    results.append({"layers": 2, "units": units, "activation": "relu",
                    "train_acc": round(float(train_acc), 3),
                    "holdout_acc": round(float(holdout_acc), 3)})

df_relu_2layer = pd.DataFrame(results)
df_relu_2layer

Epoch 0, Cost: 1.0371
Epoch 10, Cost: 1.3901
Epoch 20, Cost: 0.1805
Epoch 30, Cost: 0.0712
Epoch 40, Cost: 0.4222
Epoch 50, Cost: 1.3099
Epoch 60, Cost: 0.4229
Epoch 70, Cost: 1.1769
Epoch 80, Cost: 0.1101
Epoch 90, Cost: 0.1720
Epoch 0, Cost: 0.8486
Epoch 10, Cost: 0.1143
Epoch 20, Cost: 0.0755
Epoch 30, Cost: 0.1466
Epoch 40, Cost: 1.8686
Epoch 50, Cost: 0.1702
Epoch 60, Cost: 0.3229
Epoch 70, Cost: 0.1044
Epoch 80, Cost: 0.8478
Epoch 90, Cost: 0.0972
Epoch 0, Cost: 0.4996
Epoch 10, Cost: 0.0705
Epoch 20, Cost: 0.2934
Epoch 30, Cost: 0.0790
Epoch 40, Cost: 0.1287
Epoch 50, Cost: 0.0105
Epoch 60, Cost: 0.1512
Epoch 70, Cost: 0.1408
Epoch 80, Cost: 3.0785
Epoch 90, Cost: 3.1371
Epoch 0, Cost: 0.0608
Epoch 10, Cost: 0.1146
Epoch 20, Cost: 2.4585
Epoch 30, Cost: 0.2341
Epoch 40, Cost: 0.1936
Epoch 50, Cost: 1.2006
Epoch 60, Cost: 0.5532
Epoch 70, Cost: 0.2313
Epoch 80, Cost: 0.1024
Epoch 90, Cost: 0.3036
Epoch 0, Cost: 0.0441
Epoch 10, Cost: 0.0846
Epoch 20, Cost: 0.6422
Epoch 30, Cost: 

,layers,units,activation,train_acc,holdout_acc
0,2,2,relu,0.777,0.810
1,2,4,relu,0.796,0.795
2,2,8,relu,0.821,0.825
3,2,16,relu,0.816,0.820
4,2,32,relu,0.829,0.835
5,2,64,relu,0.830,0.845


In [13]:
# 2 layer tanh differing number of hidden units

hidden_units_grid = [2, 4, 8, 16, 32, 64]
results = []
epoches = 100
for units in hidden_units_grid:
    torch.manual_seed(1234)
    model = MulticlassClassification_2layer(units, "tanh")
    model.fit(X_train_std, y_train, n_epochs= epoches)

    train_acc = (model.predict(X_train_std) == y_train.numpy()).mean()
    holdout_acc = (model.predict(X_test_std) == y_test.numpy()).mean()

    results.append({"layers": 2, "units": units, "activation": "tanh",
                    "train_acc": round(float(train_acc), 3),
                    "holdout_acc": round(float(holdout_acc), 3)})

df_tanh_2layer = pd.DataFrame(results)
df_tanh_2layer

Epoch 0, Cost: 0.9082
Epoch 10, Cost: 0.4811
Epoch 20, Cost: 0.2652
Epoch 30, Cost: 0.2768
Epoch 40, Cost: 0.1611
Epoch 50, Cost: 1.2633
Epoch 60, Cost: 0.2324
Epoch 70, Cost: 1.4254
Epoch 80, Cost: 0.1392
Epoch 90, Cost: 0.2149
Epoch 0, Cost: 0.9452
Epoch 10, Cost: 0.2863
Epoch 20, Cost: 0.1328
Epoch 30, Cost: 0.1960
Epoch 40, Cost: 1.8460
Epoch 50, Cost: 0.1482
Epoch 60, Cost: 0.2339
Epoch 70, Cost: 0.1195
Epoch 80, Cost: 0.9004
Epoch 90, Cost: 0.1564
Epoch 0, Cost: 0.3351
Epoch 10, Cost: 0.0967
Epoch 20, Cost: 0.1204
Epoch 30, Cost: 0.0465
Epoch 40, Cost: 0.1383
Epoch 50, Cost: 0.0411
Epoch 60, Cost: 0.1118
Epoch 70, Cost: 0.0857
Epoch 80, Cost: 3.2517
Epoch 90, Cost: 3.9181
Epoch 0, Cost: 0.0423
Epoch 10, Cost: 0.1258
Epoch 20, Cost: 2.9440
Epoch 30, Cost: 0.1028
Epoch 40, Cost: 0.1665
Epoch 50, Cost: 1.1919
Epoch 60, Cost: 0.3932
Epoch 70, Cost: 0.2036
Epoch 80, Cost: 0.0822
Epoch 90, Cost: 0.2428
Epoch 0, Cost: 0.0577
Epoch 10, Cost: 0.0543
Epoch 20, Cost: 1.3573
Epoch 30, Cost: 

,layers,units,activation,train_acc,holdout_acc
0,2,2,tanh,0.785,0.795
1,2,4,tanh,0.807,0.820
2,2,8,tanh,0.815,0.820
3,2,16,tanh,0.819,0.810
4,2,32,tanh,0.844,0.830
5,2,64,tanh,0.849,0.835


In [17]:
df_all = pd.concat([df_relu_1layer, df_relu_2layer, df_tanh_1layer, df_tanh_2layer],
                   ignore_index=True)
df_all.sort_values(["activation", "layers", "units"]).reset_index(drop=True)

,layers,units,activation,train_acc,holdout_acc
0,1,2,relu,0.647,0.610
1,1,4,relu,0.777,0.795
2,1,8,relu,0.807,0.820
3,1,16,relu,0.802,0.800
4,1,32,relu,0.809,0.815
5,1,64,relu,0.818,0.825
6,2,2,relu,0.777,0.810
7,2,4,relu,0.796,0.795
8,2,8,relu,0.821,0.825
9,2,16,relu,0.816,0.820


## 3: Regression Task (25 points)

Use the dataset below to complete points 1 to 4 in the general instructions for this task.

Use as many cells as needed.

In [ ]:
from sklearn.datasets import make_regression
X, y = make_regression(n_samples=1000, n_features=2, n_informative=2, random_state=1234, noise=75)

In [ ]:
# USE AS MANY CELLS AS NEEDED

## 4. Discussion (5 points)

You created a separate class for each task and likely repeated much of the same code across implementations.

Discuss within your group how could you have leveraged inheritance to make your code more reusable and avoid duplication. Provide examples. Be specific.

YOUR TEXT HERE

## 5. Collaboration Reflection (5 points)

As a group, identify **one specific challenge or inefficiency** you encountered while working on this lab. In 3–5 sentences:

1. Briefly describe what happened and how it affected your work.
2. Identify one concrete action your group will take to avoid or better manage this issue in the next lab.

If your group encountered no major problem, identify one aspect of your collaboration that could still be improved.

YOUR TEXT HERE